# Model Group Manual

이 notebook은 `cohort grouping`의 수동 분류 출발점을 만든다.

목표:
- `brand + model_name` 기준으로 모델 인벤토리를 검토한다.
- 사람이 입력할 `major_category`를 준비한다.
- category별 체급 산정을 위한 기초 함수와 템플릿을 만든다.
- 이름 normalization은 display 값과 별개로 `_key` 컬럼을 만들어 안정적으로 관리한다.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

DATA_PATH = Path('data/cohort_grouping_input.csv').resolve()
df_raw = pd.read_csv(DATA_PATH, dtype=str)
df = df_raw.apply(lambda col: col.map(lambda value: None if isinstance(value, str) and value.strip() == '' else value))

DATA_PATH, df.shape


(PosixPath('/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/raw/models_cohort/cohort_generation/data/cohort_grouping_input.csv'),
 (7754, 9))

In [2]:
display(df[['brand', 'model_name', 'level_name', 'class_name', 'launch_price_krw', 'displacement_cc', 'body_length_mm', 'body_width_mm', 'body_height_mm']].head(20))
df.shape


,brand,model_name,level_name,class_name,launch_price_krw,displacement_cc,body_length_mm,body_width_mm,body_height_mm
0,chevrolet,뉴 볼트 EV,(전기),프리미어,41300000,0,4140,1765,1595
1,chevrolet,더 넥스트 스파크,1.0,LS,10360000,999,3595,1595,1475
2,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475
3,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475
4,chevrolet,더 넥스트 스파크,1.0,LT,11340000,999,3595,1595,1475
5,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,995,3595,1595,1475
6,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475
7,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475
8,chevrolet,더 넥스트 스파크,1.0,LTZ,11360000,995,3595,1595,1475
9,chevrolet,더 넥스트 스파크,1.0,LTZ,11340000,999,3595,1595,1475


(7754, 9)

## Missing Displacement or Body Size Rows

우선 `배기량`과 `차 사이즈` 관련 컬럼 중 하나라도 비어 있거나, 값이 `0`인 row를 따로 뽑는다.
현재 데이터에서는 실제 missing이 빈 문자열이 아니라 `0`으로 들어간 경우가 많아서, `0`도 missing으로 간주한다.

현재 기준 컬럼은 아래 네 개를 사용한다.

- `displacement_cc`
- `body_length_mm`
- `body_width_mm`
- `body_height_mm`


In [3]:
required_columns = [
    'displacement_cc',
    'body_length_mm',
    'body_width_mm',
    'body_height_mm',
]

numeric_required = df[required_columns].apply(pd.to_numeric, errors="coerce")
missing_mask = numeric_required.isna() | numeric_required.le(0)

def collect_missing_fields(row: pd.Series) -> str:
    missing_cols = [column for column in required_columns if pd.isna(row[column]) or float(row[column]) <= 0]
    return ", ".join(missing_cols)

missing_size_or_displacement_rows = df.loc[missing_mask.any(axis=1)].copy()
missing_size_or_displacement_rows["missing_fields"] = numeric_required.loc[missing_mask.any(axis=1)].apply(collect_missing_fields, axis=1)

missing_size_or_displacement_model_summary = (
    missing_size_or_displacement_rows
    .groupby(["brand", "model_name"], dropna=False)
    .agg(
        missing_row_count=("model_name", "size"),
        missing_field_examples=("missing_fields", lambda s: ", ".join(list(dict.fromkeys(str(v) for v in s.dropna()))[:10])),
    )
    .reset_index()
    .sort_values(["missing_row_count", "brand", "model_name"], ascending=[False, True, True])
)

df_filtered = df.loc[~missing_mask.any(axis=1)].copy()
df = df_filtered.copy()

print("original rows:", len(df_raw))
print("removed rows:", len(missing_size_or_displacement_rows))
print("filtered rows:", len(df_filtered))
print("affected models:", len(missing_size_or_displacement_model_summary))

display(missing_size_or_displacement_model_summary.head(50))

display(df.head(20))

df.shape


original rows: 7754
removed rows: 1845
filtered rows: 5909
affected models: 186


,brand,model_name,missing_row_count,missing_field_examples
150,kia,봉고3,227,"displacement_cc, body_length_mm, body_width_mm..."
80,hyundai,포터2,219,"displacement_cc, body_length_mm, body_width_mm..."
78,hyundai,파비스,74,"displacement_cc, body_length_mm, body_width_mm..."
138,kia,더 뉴 봉고3,63,"displacement_cc, body_length_mm, body_width_mm..."
77,hyundai,트라고 엑시언트,51,"displacement_cc, body_length_mm, body_width_mm..."
16,hyundai,e마이티,50,"displacement_cc, body_length_mm, body_width_mm..."
76,hyundai,트라고,46,"displacement_cc, body_length_mm, body_width_mm..."
21,hyundai,그랜드스타렉스,42,"displacement_cc, body_length_mm, body_width_mm..."
65,hyundai,엑시언트 프로,42,"displacement_cc, body_length_mm, body_width_mm..."
64,hyundai,엑시언트,38,"displacement_cc, body_length_mm, body_width_mm..."


,brand,model_name,level_name,class_name,launch_price_krw,displacement_cc,body_length_mm,body_width_mm,body_height_mm
1,chevrolet,더 넥스트 스파크,1.0,LS,10360000,999,3595,1595,1475
2,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475
3,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475
4,chevrolet,더 넥스트 스파크,1.0,LT,11340000,999,3595,1595,1475
5,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,995,3595,1595,1475
6,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475
7,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475
8,chevrolet,더 넥스트 스파크,1.0,LTZ,11360000,995,3595,1595,1475
9,chevrolet,더 넥스트 스파크,1.0,LTZ,11340000,999,3595,1595,1475
10,chevrolet,더 넥스트 스파크,1.0,LTZ,11340000,999,3595,1595,1475


(5909, 9)

## Model Category Mapping Goal

이 단계의 목표는 `brand + model_name` 단위 매핑 테이블을 만드는 것이다.
즉 trim이 달라도 같은 모델이면 같은 차체/카테고리로 묶는다는 가정을 쓴다.

최종적으로 만들 테이블 컬럼은 다음과 같다.
- `brand`
- `model_name`
- `major_category`
- `market_family`


In [4]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from cohort_mapping import (
    CohortAgentConfig,
    VehicleCategoryMappingInput,
    VehicleCategoryMappingOutput,
    apply_mapping_table,
    build_mapping_requests,
    dataframe_to_mapping_inputs,
    derive_market_family,
    get_input_schema,
    get_output_schema,
    get_pending_requests,
    load_mapping_table,
    mapping_outputs_to_frame,
    run_category_mapping_sync,
)
from cohort_mapping.config import is_agent_env_ready


ModuleNotFoundError: No module named 'dotenv'

## Input / Output Schema

schema는 단순하게 `brand + model_name`에 대한 category mapping만 다룬다.


In [ ]:
input_schema_fields = pd.DataFrame(
    [
        {
            "field": name,
            "annotation": str(field.annotation),
            "required": field.is_required(),
        }
        for name, field in VehicleCategoryMappingInput.model_fields.items()
    ]
)

output_schema_fields = pd.DataFrame(
    [
        {
            "field": name,
            "annotation": str(field.annotation),
            "required": field.is_required(),
        }
        for name, field in VehicleCategoryMappingOutput.model_fields.items()
    ]
)

display(input_schema_fields)
display(output_schema_fields)


## Build Mapping Requests

`brand + model_name` 기준으로 request table을 만든다.
`class_name`과 `level_name`은 examples로만 전달한다.


In [ ]:
requests_df = build_mapping_requests(df)
requests_path = DATA_PATH.parent / "model_category_mapping_requests.csv"
requests_df.to_csv(requests_path, index=False)

display(requests_df.head(30))
requests_path, requests_df.shape


## Load Mapping Table and Skip Existing Rows

이미 매핑된 `brand + model_name`이 있으면 스킵하고, 없는 것만 agent 호출 대상으로 남긴다.


In [ ]:
mapping_table_path = DATA_PATH.parent / "model_category_mapping_table.csv"
mapping_table_df = load_mapping_table(mapping_table_path)
pending_requests_df = get_pending_requests(requests_df, mapping_table_df)

print("existing mapping rows:", len(mapping_table_df))
print("pending requests:", len(pending_requests_df))

display(mapping_table_df.head(20))
display(pending_requests_df.head(20))


## Run Category Agent Iteratively

이 셀은 notebook에서 순차적으로 agent를 호출한다.
각 결과는 즉시 `model_category_mapping_table.csv`에 append/save 하므로 중간에 끊겨도 이어서 돌릴 수 있다.


In [ ]:
RUN_AGENT = True
RUN_LIMIT = 5

if not is_agent_env_ready():
    print('OPENROUTER_API_KEY is missing. Fill cohort_generation/.env first.')
elif not RUN_AGENT:
    print('Set RUN_AGENT = True to execute the category mapping agent.')
else:
    config = CohortAgentConfig.from_env()
    rows_to_run = pending_requests_df.head(RUN_LIMIT).copy()
    appended_rows = []
    for request in dataframe_to_mapping_inputs(rows_to_run):
        result = run_category_mapping_sync(request, config)
        result_row = result.model_dump(mode="json")
        result_row["market_family"] = derive_market_family(result_row["major_category"])
        appended_rows.append(result_row)
        mapping_table_df = pd.concat([mapping_table_df, pd.DataFrame([result_row])], ignore_index=True)
        mapping_table_df = mapping_table_df.drop_duplicates(subset=["brand", "model_name"], keep="last")
        mapping_table_df.to_csv(mapping_table_path, index=False)

    appended_df = pd.DataFrame(appended_rows)
    display(appended_df)
    display(mapping_table_df.tail(20))
    mapping_table_path


## Apply Mapping Table to Cohort Input

완성된 mapping table을 `cohort_grouping_input.csv`에 붙여서 category가 포함된 최종 파일을 만든다.


In [ ]:
mapping_table_df = load_mapping_table(mapping_table_path)
cohort_grouping_with_category = apply_mapping_table(df, mapping_table_df)
output_path = DATA_PATH.parent / "cohort_grouping_with_category.csv"
cohort_grouping_with_category.to_csv(output_path, index=False)

print("mapped rows:", cohort_grouping_with_category[["major_category", "market_family"]].dropna().shape[0])
display(cohort_grouping_with_category.head(30))
output_path
